<a href="https://colab.research.google.com/github/Chhoulong-Banh/cyclistic-bike-share-analysis/blob/main/cyclistic_analysis_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚴 Cyclistic Bike-Share: Pipeline Engine
**Project Goal:** Automated Data Ingestion & Quality Control
**Environment:** Google Colab + Azure PostgreSQL

## Notebook Table of Contents
1. **[Config]** Environment & Drive Mount
2. **[Diagnostic Profiler]** Schema Drift Detection & Metadata Generation
3. **[Contract Validator]** Data Enforcement & Quarantine Routing
4. **[Analysis]** (Future Phase)

---
## Pipeline Version History
- **v1.0.0:** Infrastructure & Connection Handshake (Sprint 1)
- **v1.1.0:** Iterative Profiling & Schema Auto-Discovery (Sprint 2)
- **v1.2.0:** Contract Validation & Quarantine Routing (Sprint 3)

In [ ]:
# Cell 1: Environment & Credentials Setup
# We use 'userdata' to securely pull sensitive passwords that shouldn't be hardcoded.
from google.colab import userdata
from sqlalchemy import create_engine
import os

# Securely retrieve database credentials stored in the Colab "Secrets" vault (the key icon)
# This prevents your passwords from being visible in your notebook or GitHub repository.
host = userdata.get('AZURE_HOST')
db   = userdata.get('AZURE_DB')
user = userdata.get('AZURE_USER')
password = userdata.get('AZURE_PASS')

# Create the SQLAlchemy "engine," which acts as the bridge (connector) to your Azure database.
# SSL is required by Azure for secure, encrypted connections.
connection_string = f"postgresql+psycopg2://{user}:{password}@{host}/{db}?sslmode=require"
engine = create_engine(connection_string)

# Perform a "handshake" test to ensure the connection is successful.
try:
    with engine.connect() as conn:
        print("Successfully connected to Azure PostgreSQL!")
except Exception as e:
    # If there is an error (like a wrong password or network issue), this explains why.
    print(f"Connection failed: {e}")

In [ ]:
# Cell 2: Diagnostic Profiler Part 1 - Google Drive Mount
# --- STEP 1: Cloud Workspace Connection & Authentication ---
# We import the built-in Google Colab drive module and the Python operating system (os) package.
# This allows us to work with files and directories stored securely in the cloud.
from google.colab import drive
import os

# Establish a secure cryptographic handshake to mount your personal Google Drive.
# This step makes your cloud drive folders safely accessible as local directories within the Colab kernel.
drive.mount('/content/drive')

# Establish the master home base directory for this Capstone project.
# Centralizing this path makes it easy to update if your file storage layout changes in the future.
BASE_DIR = '/content/drive/MyDrive/Demo/CaseStudy1-CyclisticBikes/'
RAW_DATA_PATH = os.path.join(BASE_DIR, 'OriginalDataFile-2025/')

# Pre-Flight Validation Check: Verify that the source data directory is actually reachable.
# Programmatically checking paths before running loops prevents silent runtime failures.
if os.path.exists(RAW_DATA_PATH):
    print(f"Connection established! Data detected at: {RAW_DATA_PATH}")
else:
    # A clear error flag helps track folder misalignments immediately during deployment.
    print(f"ALERT: The folder {RAW_DATA_PATH} was not found. Check your Drive path.")

In [ ]:
# Cell 3: Diagnostic Profiler Part 2 - Complete Stateful Diagnostic Profiler Engine
import pandas as pd
import json
import os
import uuid
from datetime import datetime
from pathlib import Path

# --- Configuration & Workspace Paths ---
# We define clean, absolute folder paths across Google Drive to keep our pipeline organized.
BASE_DIR = '/content/drive/MyDrive/Demo/CaseStudy1-CyclisticBikes/'
RAW_DATA_PATH = os.path.join(BASE_DIR, 'OriginalDataFile-2025/')
SCHEMA_PATH = os.path.join(BASE_DIR, 'master_schema.json')
PY_SCHEMA_PATH = os.path.join(BASE_DIR, 'py_schema.json')
AUDIT_LOG_PATH = os.path.join(BASE_DIR, 'pipeline_audit.log')
TYPE_RULES_PATH = os.path.join(BASE_DIR, 'type_rules.json')

# Generate a unique 8-character token to track this specific execution runtime session.
SESSION_ID = str(uuid.uuid4())[:8]

# Diagnostic Arrays: Used to capture runtime anomalies or validation notes for the final summary map.
run_errors = []
run_warnings = []

# Fallback Blueprint Matrix: Enforces strict data-type mappings if an external rule book is missing.
DEFAULT_TYPE_RULES = {
    "default_pandas_to_sql_mapping": {
        'int64': 'INTEGER',
        'float64': 'NUMERIC(10, 6)',
        'bool': 'BOOLEAN',
        'datetime64[ns]': 'TIMESTAMP',
        'object': 'VARCHAR(255)'
    },
    "column_name_to_sql_type_overrides": {
        "lat": "NUMERIC(10, 6)",
        "lng": "NUMERIC(10, 6)",
        "price": "NUMERIC(12, 2)"
    },
    "read_csv_explicit_dtypes": {
        'start_station_name': 'str',
        'start_station_id': 'str',
        'end_station_name': 'str',
        'end_station_id': 'str'
    }
}

def log_event(message, level="INFO"):
    """Central auditing utility that logs pipeline events to a file and prints them live."""
    global run_errors, run_warnings
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_entry = f"[{timestamp}] [RUN:{SESSION_ID}] [{level}] {message}"
    
    # Open the log text file in 'append' mode to keep a historical record without overwriting past entries.
    with open(AUDIT_LOG_PATH, 'a') as f:
        f.write(log_entry + "\n")
    print(log_entry)
    
    # Categorize anomalies so they can be securely surfaced in the final execution report.
    if level == "ERROR" or level == "CRITICAL":
        run_errors.append({"timestamp": timestamp, "message": message, "level": level})
    elif level == "WARNING":
        run_warnings.append({"timestamp": timestamp, "message": message, "level": level})

def load_type_rules(path):
    """Safely extracts your structural translation rules from cold database files."""
    if not os.path.exists(path):
        log_event(f"Type rules file not found at {path}. Using default rules.", "WARNING")
        return DEFAULT_TYPE_RULES
    try:
        with open(path, 'r') as f:
            rules = json.load(f)
        # Ensure all required blueprint properties exist in the custom file.
        for key in DEFAULT_TYPE_RULES:
            if key not in rules:
                log_event(f"Missing key '{key}' in {os.path.basename(path)}. Using default for this section.", "WARNING")
                rules[key] = DEFAULT_TYPE_RULES[key]
        return rules
    except json.JSONDecodeError:
        log_event(f"{os.path.basename(path)} is not valid JSON. Using default rules.", "CRITICAL")
        return DEFAULT_TYPE_RULES
    except Exception as e:
        log_event(f"Error loading {os.path.basename(path)}: {e}. Using default rules.", "CRITICAL")
        return DEFAULT_TYPE_RULES

# --- Initialization Gate ---
# Programmatically ensure the project directory tree is active before writing files.
os.makedirs(BASE_DIR, exist_ok=True)
with open(TYPE_RULES_PATH, 'w') as f:
    json.dump(DEFAULT_TYPE_RULES, f, indent=4)
log_event(f"Ensured '{TYPE_RULES_PATH}' exists with default type rules.", "INFO")

# Parse active settings maps into local configuration variables
TYPE_RULES = load_type_rules(TYPE_RULES_PATH)
SQL_TYPE_MAP_BASE = TYPE_RULES["default_pandas_to_sql_mapping"]
COLUMN_NAME_SQL_TYPE_OVERRIDES = TYPE_RULES["column_name_to_sql_type_overrides"]

# Convert textual type strings from the JSON definition into executable Python type objects.
EXPLICIT_READ_DTYPES = {}
for col, dtype_str in TYPE_RULES["read_csv_explicit_dtypes"].items():
    try:
        EXPLICIT_READ_DTYPES[col] = eval(dtype_str)
    except NameError:
        log_event(f"Invalid dtype string '{dtype_str}' for column '{col}' in type_rules.json. Skipping.", "WARNING")
        EXPLICIT_READ_DTYPES[col] = object

def bump_version(ver):
    """Programmatically increments a semantic version string (e.g., '1.0.4' to '1.0.5')."""
    try:
        major, minor, patch = map(int, ver.split('.'))
        return f"{major}.{minor}.{patch + 1}"
    except:
        return "1.0.0"

def get_registry(path, is_master_schema=False):
    """Initializes or loads a historical schema log registry to detect data-integrity drift over time."""
    if not os.path.exists(path):
        data = {"active_schema": {}, "revision_history": []} if is_master_schema else {"revision_history": []}
        with open(path, 'w') as f:
            json.dump(data, f, indent=4)
        return data
    try:
        with open(path, 'r') as f:
            loaded_content = json.load(f)

        # Re-verify and correct file contents if a past execution corrupted the layout.
        if is_master_schema:
            if not (isinstance(loaded_content, dict) and "active_schema" in loaded_content and "revision_history" in loaded_content):
                log_event(f"{os.path.basename(path)} format was unexpected. Rebuilding structure.", "WARNING")
                loaded_content = {"active_schema": loaded_content if isinstance(loaded_content, dict) else {}, "revision_history": []}
        else:
            if not (isinstance(loaded_content, dict) and "revision_history" in loaded_content):
                log_event(f"{os.path.basename(path)} format was unexpected. Rebuilding structure.", "WARNING")
                loaded_content = {"revision_history": []}

        if is_master_schema:
            # Self-Healing Layer 1: Correct station IDs mistakenly parsed as numbers due to missing values.
            for col_name, expected_pandas_type_class in EXPLICIT_READ_DTYPES.items():
                if col_name in loaded_content["active_schema"] and \
                   loaded_content["active_schema"][col_name] in [SQL_TYPE_MAP_BASE['float64'], 'DOUBLE PRECISION'] and \
                   expected_pandas_type_class == str:
                    loaded_content["active_schema"][col_name] = SQL_TYPE_MAP_BASE['object']
                    log_event(f"Corrected type for '{col_name}' in loaded master_schema from float-based to '{SQL_TYPE_MAP_BASE['object']}'.", "WARNING")

            # Self-Healing Layer 2: Correct date columns mistakenly saved as generic text.
            DATETIME_COLS = ['started_at', 'ended_at']
            for col_name in DATETIME_COLS:
                if col_name in loaded_content["active_schema"] and \
                   loaded_content["active_schema"][col_name] == SQL_TYPE_MAP_BASE['object'] and \
                   SQL_TYPE_MAP_BASE['datetime64[ns]'] == 'TIMESTAMP':
                    loaded_content["active_schema"][col_name] = SQL_TYPE_MAP_BASE['datetime64[ns]']
                    log_event(f"Corrected type for '{col_name}' in loaded master_schema from '{SQL_TYPE_MAP_BASE['object']}' to '{SQL_TYPE_MAP_BASE['datetime64[ns]']}'.", "WARNING")

            # Self-Healing Layer 3: Enforce strict precision definitions for geo-coordinates and financial indicators.
            FLOAT_RELATED_COLS = ['start_lat', 'start_lng', 'end_lat', 'end_lng', 'price']
            for col_name in FLOAT_RELATED_COLS:
                expected_sql_type = SQL_TYPE_MAP_BASE['float64']
                for pattern, sql_override_type in COLUMN_NAME_SQL_TYPE_OVERRIDES.items():
                    if pattern == col_name:
                        expected_sql_type = sql_override_type
                        break

                if col_name in loaded_content["active_schema"] and \
                   loaded_content["active_schema"][col_name] == 'DOUBLE PRECISION' and \
                   expected_sql_type == 'NUMERIC(10, 6)':
                    loaded_content["active_schema"][col_name] = expected_sql_type
                    log_event(f"Corrected type for '{col_name}' in loaded master_schema from 'DOUBLE PRECISION' to '{expected_sql_type}'.", "WARNING")
                elif col_name in loaded_content["active_schema"] and \
                     loaded_content["active_schema"][col_name] == 'DOUBLE PRECISION' and \
                     expected_sql_type == 'NUMERIC(12, 2)':
                    loaded_content["active_schema"][col_name] = expected_sql_type
                    log_event(f"Corrected type for '{col_name}' in loaded master_schema from 'DOUBLE PRECISION' to '{expected_sql_type}'.", "WARNING")

        return loaded_content
    except json.JSONDecodeError:
        log_event(f"{os.path.basename(path)} is not valid JSON. Initializing new structure.", "CRITICAL")
        return {"active_schema": {}, "revision_history": []} if is_master_schema else {"revision_history": []}
    except Exception as e:
        log_event(f"Critical failure loading {os.path.basename(path)}: {e}. Initializing new structure.", "CRITICAL")
        return {"active_schema": {}, "revision_history": []} if is_master_schema else {"revision_history": []}

# --- STAGE 1: Session Initialization ---
profiler_start_time = datetime.now()
log_event("SESSION_START: Profiler execution opened.")
master_registry = get_registry(SCHEMA_PATH, is_master_schema=True)
py_registry = get_registry(PY_SCHEMA_PATH)
files = list(Path(RAW_DATA_PATH).glob('*.csv'))

# Ingestion Guard: Freeze execution if no files exist to prevent unhandled loop crashes.
if not files:
    log_event("CRITICAL: No files found for profiling.", "CRITICAL")
    raise FileNotFoundError("Check RAW_DATA_PATH")

print(f"Found {len(files)} files to profile. Starting scan...")

# --- STAGE 2: Forensic Inference & Schema Drift Validation ---
drift_detected = False
compiled_drift_details = {}

file_py_types = {}
file_sql_types = {}

# Loop iteratively through every CSV source log inside the target folder.
for file in files:
    log_event(f"FILE_START: {file.name}")
    try:
        # Load a 100-row sample snapshot of the file to discover structural schema constraints efficiently.
        df = pd.read_csv(file, nrows=100, dtype=EXPLICIT_READ_DTYPES, parse_dates=['started_at', 'ended_at'])

        # Print the detailed metadata snapshot cleanly to the active terminal window.
        print(f"\n--- Profiling: {file.name} ---")
        print(f"Column Structure: {list(df.columns)}")
        print(f"Data Types: \n{df.dtypes}")

        # Map pandas data types to generic database properties.
        py_types = {col: str(df[col].dtype) for col in df.columns}
        sql_types = {col: SQL_TYPE_MAP_BASE.get(str(df[col].dtype), SQL_TYPE_MAP_BASE['object']) for col in df.columns}

        # Apply specific overrides for fields requiring precision (e.g., coordinates).
        for col_name in sql_types:
            for pattern, sql_override_type in COLUMN_NAME_SQL_TYPE_OVERRIDES.items():
                if pattern in col_name:
                    sql_types[col_name] = sql_override_type
                    break

        file_py_types = py_types
        file_sql_types = sql_types

        # Guard Rule: Verify current file structure against our established master registry blueprint.
        if master_registry['active_schema']:
            if sql_types != master_registry['active_schema']:
                drift_detected = True
                drift_details_current_file = {}
                # Trace exactly where the structure diverged (mismatches, missing keys, new parameters).
                for col_name, expected_dtype in master_registry['active_schema'].items():
                    if col_name in sql_types and sql_types[col_name] != expected_dtype:
                        drift_details_current_file[col_name] = f"Type mismatch: Expected {expected_dtype}, Got {sql_types[col_name]}"
                    elif col_name not in sql_types:
                        drift_details_current_file[col_name] = "Column missing from file"
                for col_name in sql_types:
                    if col_name not in master_registry['active_schema']:
                        drift_details_current_file[col_name] = "New column in file"

                compiled_drift_details[file.name] = drift_details_current_file
                log_event(f"CRITICAL: Schema drift detected in {file.name}. Discrepancies: {drift_details_current_file}", "CRITICAL")
            else:
                log_event(f"INFO: {file.name} validated successfully against master_registry.", "INFO")
        else:
            # If the schema file is blank, use the first file's layout as our baseline configuration.
            master_registry['active_schema'] = sql_types
            log_event(f"INFO: Master schema established from {file.name} (first file in Discovery mode).", "INFO")

        log_event(f"FILE_COMPLETE: {file.name}")
    except Exception as e:
        log_event(f"ERROR: Processing failed for {file.name}: {e}", "ERROR")
        drift_detected = True

print("\nProfiling complete. Your data structure has been successfully mapped.")

# --- STAGE 3: Synchronized Versioning & Incremental Commit ---
# Compute the updated minor version patch token.
new_ver = bump_version(master_registry['revision_history'][0]['version'] if master_registry['revision_history'] else "1.0.0")
event_tag = "FAILURE_HALTED" if drift_detected else "SUCCESS"

# Log the current structural configuration object into the master historical JSON schema files.
if master_registry['active_schema']:
    master_registry['revision_history'].insert(0, {
        "version": new_ver,
        "ts": datetime.now().isoformat(),
        "event": event_tag,
        "schema": master_registry['active_schema']
    })
else:
    log_event("WARNING: No active_schema established for master_registry; skipping commit to revision history.", "WARNING")

if file_py_types:
    py_registry['revision_history'].insert(0, {
        "version": new_ver,
        "ts": datetime.now().isoformat(),
        "event": event_tag,
        "schema": file_py_types
    })
else:
    log_event("WARNING: No Python schema observations to record as no files were processed successfully.", "WARNING")

# Write changes back to persistent storage to lock in the new version history metadata.
with open(SCHEMA_PATH, 'w') as f:
    json.dump(master_registry, f, indent=4)
with open(PY_SCHEMA_PATH, 'w') as f:
    json.dump(py_registry, f, indent=4)

# --- STAGE 4: Summary Telemetry Export ---
profiler_end_time = datetime.now()
duration = str(profiler_end_time - profiler_start_time)

run_info = {
    "session_id": SESSION_ID,
    "start_time": profiler_start_time.isoformat(),
    "end_time": profiler_end_time.isoformat(),
    "duration": duration,
    "status": event_tag,
    "version": new_ver,
    "errors": run_errors,
    "warnings": run_warnings,
    "drift_details": compiled_drift_details
}

print("\n--- Profiler Run Summary ---")
display(run_info)

log_event(f"SESSION_END: Profiler closed. Version locked: v{new_ver}. Status: {event_tag}")

In [ ]:
# Cell 4: Automated Schema Bootstrapping (Dynamic & Reusable)
# Provisions Azure tables based on Sprint 2 schema and defined Primary Keys.

import json
from sqlalchemy import text

# --- Configuration Mapping (Semantic Overrides) ---
# Decouples business logic (Primary Keys) from technical infrastructure (Schema Builder)
TABLE_PRIMARY_KEYS = {
    "stg_cyclistic_trips": "ride_id",
    "error_quarantine_log": "quarantine_id",
    "job_log": "file_name"
}

# 1. Load the blueprint (Sprint 2 Output)
with open(SCHEMA_PATH, 'r') as f:
    master_registry = json.load(f)

# Extract schema definition
schema_definition = master_registry.get("active_schema", {})

def generate_create_table_sql(table_name, schema_dict, pk_map):
    """
    Translates JSON schema into a PostgreSQL CREATE TABLE string with dynamic PK enforcement.
    """
    columns = [f"{col} {dtype}" for col, dtype in schema_dict.items()]
    
    # Check if a Primary Key is defined for this table in our semantic map
    pk_column = pk_map.get(table_name)
    if pk_column:
        columns.append(f"PRIMARY KEY ({pk_column})")
        
    return f"CREATE TABLE IF NOT EXISTS {table_name} ({', '.join(columns)});"

# 2. Programmatic Provisioning
with engine.connect() as conn:
    # A. Provision Staging Table
    # The staging table now enforces PRIMARY KEY (ride_id), enabling database-native deduplication.
    create_stg_sql = generate_create_table_sql('stg_cyclistic_trips', schema_definition, TABLE_PRIMARY_KEYS)
    conn.execute(text(create_stg_sql))
    
    # B. Provision Quarantine Table
    # Now explicitly includes the quarantine_id as PK, allowing for audit lineage.
    create_quarantine_sql = generate_create_table_sql(
        'error_quarantine_log', 
        {
            "quarantine_id": "UUID",
            "ride_id": "VARCHAR(255)",
            "file_name": "VARCHAR(255)",
            "failed_column": "VARCHAR(100)",
            "error_justification": "TEXT",
            "row_data": "JSONB",
            "status": "VARCHAR(20)",
            "created_at": "TIMESTAMP DEFAULT CURRENT_TIMESTAMP"
        }, 
        TABLE_PRIMARY_KEYS
    )
    conn.execute(text(create_quarantine_sql))
    
    # C. Provision Operational Job Log
    # Essential state-tracking table for pipeline idempotency.
    create_job_log_sql = generate_create_table_sql(
        'job_log', 
        {
            "file_name": "VARCHAR(255)",
            "processed_at": "TIMESTAMP",
            "status": "VARCHAR(20)",
            "total_records": "INTEGER",
            "passed_records": "INTEGER",
            "quarantined_records": "INTEGER"
        }, 
        TABLE_PRIMARY_KEYS
    )
    conn.execute(text(create_job_log_sql))
    
    conn.commit()
    log_event("Schema provisioning complete: Infrastructure tables verified/initialized.", "INFO")

In [ ]:
# Cell 5: Reusable, Immutable Ingestion Engine (v8.1)
# Prerequisites: Assumes 'engine', 'master_registry', 'files', and 'log_event' are active.

import uuid
import numpy as np
import pandas as pd
from datetime import datetime
from sqlalchemy import text
from sqlalchemy.exc import DataError

# --- PIPELINE CONFIGURATION (REUSABILITY LAYER) ---
# Modify these variables to deploy this engine to any other project
PIPELINE_CONFIG = {
    "TARGET_TABLE": "stg_cyclistic_trips",
    "ERROR_LOG_TABLE": "error_quarantine_log",
    "JOB_LOG_TABLE": "job_log",
    "PRIMARY_KEY_COL": "ride_id"
}

# 1. Pipeline Structural Dependency Verification
try:
    ACTIVE_SCHEMA_BLUEPRINT = master_registry.get("active_schema", {})
    log_event("DEPENDENCY_SYNC: Active schema blueprint safely bound for database-native deduplication loops.")
except NameError:
    raise NameError("CRITICAL: 'master_registry' object not found in memory. Please execute Sprints 1 & 2 first.")


def validate_and_route(chunk: pd.DataFrame, file_name: str, active_schema: dict, config: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Validates a data chunk against data contract rules dynamically based on configuration.
    """
    is_valid_mask = np.ones(len(chunk), dtype=bool)
    quarantine_records = []
    chunk_reset = chunk.reset_index(drop=True)
    pk_col = config["PRIMARY_KEY_COL"]

    # 1. Row-Level Structural Verification Gate
    for idx, row in chunk_reset.iterrows():
        # --- Primary Key Empty Space Check ---
        r_id = str(row.get(pk_col, '')).strip()
        if not r_id or r_id in ['nan', 'None', '']:
            is_valid_mask[idx] = False
            quarantine_records.append({
                "quarantine_id": str(uuid.uuid4()),
                config["PRIMARY_KEY_COL"]: "MISSING_ID",
                "file_name": file_name,
                "failed_column": pk_col,
                "error_justification": f"Contract Breach: Primary key anchor field '{pk_col}' is missing or blank.",
                "row_data": row.to_json(),
                "status": "PENDING",
                "created_at": datetime.now()
            })
            continue 

        # --- Datatype Structural Verification ---
        for col, expected_sql_type in active_schema.items():
            val = row.get(col)
            if 'NUMERIC' in expected_sql_type or 'INTEGER' in expected_sql_type:
                try:
                    if pd.notnull(val) and str(val).strip() != '':
                        float(val)
                except (ValueError, TypeError):
                    is_valid_mask[idx] = False
                    quarantine_records.append({
                        "quarantine_id": str(uuid.uuid4()),
                        config["PRIMARY_KEY_COL"]: r_id,
                        "file_name": file_name,
                        "failed_column": col,
                        "error_justification": f"Contract Breach: Value '{val}' violates column database type {expected_sql_type}.",
                        "row_data": row.to_json(),
                        "status": "PENDING",
                        "created_at": datetime.now()
                    })
                    break 

    # 2. Extract Clean DataFrame and explicitly normalize types
    clean_df = chunk_reset[is_valid_mask].copy()
    if not clean_df.empty:
        for col, sql_type in active_schema.items():
            if col in clean_df.columns:
                if 'NUMERIC' in sql_type or 'INTEGER' in sql_type:
                    clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce').fillna(0)
                elif 'TIMESTAMP' in sql_type:
                    clean_df[col] = pd.to_datetime(clean_df[col], errors='coerce')

    quarantine_df = pd.DataFrame(quarantine_records)
    return clean_df, quarantine_df


# --- Main Processing Ingestion Loop ---
log_event("PIPELINE_START: Initializing abstract, append-only transaction stream.")

for file in files:
    # A. Orchestration Check: Did this file already complete successfully in a prior run?
    with engine.connect() as conn:
        chk_query = text(f"SELECT 1 FROM {PIPELINE_CONFIG['JOB_LOG_TABLE']} WHERE file_name = :f AND status = 'SUCCESS'")
        if conn.execute(chk_query, {"f": file.name}).fetchone():
            log_event(f"ORCHESTRATION: File '{file.name}' previously registered as SUCCESS. Skipping.", "INFO")
            continue

    # B. Immutable Audit Initialization: Generate unique Run ID and log IN-PROGRESS
    job_run_id = str(uuid.uuid4())
    log_event(f"STATE_TRANSITION: Starting new execution run [{job_run_id[:8]}] for target dataset: {file.name}")
    
    with engine.connect() as conn:
        # Schema agnostic insert using the configuration names, strictly passing the exact job_run_id
        conn.execute(text(f"""
            INSERT INTO {PIPELINE_CONFIG['JOB_LOG_TABLE']} 
            (job_id, file_name, processed_at, status, total_records, passed_records, quarantined_records)
            VALUES (:jid, :f, :ts, 'IN-PROGRESS', 0, 0, 0);
        """), {"jid": job_run_id, "f": file.name, "ts": datetime.now()})
        conn.commit()

    passed_cnt = 0
    quarantined_cnt = 0
    run_corrupted = False

    try:
        # C. Slid-Window Streaming File Data Parse
        for chunk in pd.read_csv(file, chunksize=50000, keep_default_na=False):
            clean_batch, dirty_batch = validate_and_route(chunk, file.name, ACTIVE_SCHEMA_BLUEPRINT, PIPELINE_CONFIG)
            
            # D. Database-Native Virtual Deduplication (The NOT EXISTS Pattern)
            if not clean_batch.empty:
                # 1. Write the chunk to an isolated temporary location
                temp_staging_name = f"tmp_stage_{str(uuid.uuid4())[:8]}"
                clean_batch.to_sql(temp_staging_name, engine, if_exists='replace', index=False)
                
                # 2. Insert only rows where the Primary Key does NOT already exist in the target table
                cols = ", ".join(clean_batch.columns)
                pk = PIPELINE_CONFIG['PRIMARY_KEY_COL']
                target = PIPELINE_CONFIG['TARGET_TABLE']
                
                insert_not_exists_query = f"""
                    INSERT INTO {target} ({cols})
                    SELECT {cols} FROM {temp_staging_name} AS src
                    WHERE NOT EXISTS (
                        SELECT 1 FROM {target} AS dest
                        WHERE dest.{pk} = CAST(src.{pk} AS VARCHAR)
                    );
                """
                
                try:
                    with engine.connect() as conn:
                        result = conn.execute(text(insert_not_exists_query))
                        conn.execute(text(f"DROP TABLE IF EXISTS {temp_staging_name};"))
                        conn.commit()
                        passed_cnt += result.rowcount # Accurately tracks rows actually inserted, ignoring duplicates
                except DataError as driver_err:
                    log_event(f"DRIVER_EXCEPTION: Data format rejection at driver layer: {driver_err}", "ERROR")
                    run_corrupted = True
                    break

            # E. Persistent Forensic Quarantine Tracking Layer Load
            if not dirty_batch.empty:
                dirty_batch.to_sql(PIPELINE_CONFIG['ERROR_LOG_TABLE'], engine, if_exists='append', index=False)
                quarantined_cnt += len(dirty_batch)

        # F. State Resolution
        final_status = 'FAILED_DATA_ERROR' if run_corrupted else 'SUCCESS'
        
        # We target ONLY the exact job_id we created in Step B, avoiding any previous crash artifacts
        with engine.connect() as conn:
            conn.execute(text(f"""
                UPDATE {PIPELINE_CONFIG['JOB_LOG_TABLE']} SET 
                    processed_at = :ts, status = :status,
                    total_records = :tot, passed_records = :p, quarantined_records = :q
                WHERE job_id = :jid
            """), {
                "ts": datetime.now(), 
                "status": final_status,
                "tot": (passed_cnt + quarantined_cnt), 
                "p": passed_cnt, "q": quarantined_cnt, "jid": job_run_id
            })
            conn.commit()
            
        print(f"📊 [PIPELINE TRANSACTION] {file.name} | Status: {final_status} | Safely Inserted: {passed_cnt:,} | Quarantined: {quarantined_cnt:,}")

    except Exception as hardware_crash:
        log_event(f"RUN_CRASH: Fatal block disruption on {file.name}. Run preserved as IN-PROGRESS. Reason: {hardware_crash}", "CRITICAL")